# Qwen APPS: published SAE sparse-probing observer

**Question.** On the frozen Qwen2.5-7B ControlArena/APPS panel, does a published sparse-autoencoder feature observer improve fixed-budget safety decisions over matched raw-residual and SAE-reconstruction controls?

The method is fixed before seeing its result: the pinned `andyrdt` layer-19 BatchTopK SAE, Kantamneni et al.'s class-mean feature ranking and top-feature L1 logistic probe, and the existing ObserverBench 2% audit policy. The run reports the locked-test decision metrics, reconstruction quality, context-length strata, rare-prevalence sensitivity, and paired problem bootstrap intervals.

Experiments designed/concieved by Vijay Erramilli. Code written by Vijay Erramilli and Codex

> The SAE checkpoint is 3.76 GB. Encoding runs once and writes a resumable, pickle-free cache. This notebook does not reload Qwen or repeat Qwen inference.


## 1. Install the frozen experiment

Use a Colab GPU runtime. The default repository and commit may be overridden with environment variables when reproducing a tagged release.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = os.environ.get(
    'OBSERVERBENCH_REPO_URL',
    'https://github.com/kwisatzh/observerbench.git',
)
REPO_REVISION = os.environ.get('OBSERVERBENCH_REVISION', 'main')
REPO_ROOT = Path(os.environ.get('OBSERVERBENCH_REPO_ROOT', '/content/observerbench'))

if not (REPO_ROOT / '.git').exists():
    subprocess.run(
        ['git', 'clone', '--depth=1', '--branch', REPO_REVISION, REPO_URL, str(REPO_ROOT)],
        check=True,
    )
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_ROOT}[sae]'],
    check=True,
)
print('ObserverBench source:', REPO_ROOT)


## 2. Stage the frozen activation cache

The 56 MB cache contains the already-computed neutral-context layer activations. Point `OBSERVERBENCH_APPS_CACHE` at a local or Drive copy. If no path is supplied in Colab, the cell opens a file picker. The runner verifies the file's frozen SHA-256 before using it.


In [ ]:
CACHE_TARGET = (
    REPO_ROOT
    / 'results/revision/ai_control/controlarena_apps_qwen_followup_v1/followup_cache.npz'
)
CACHE_TARGET.parent.mkdir(parents=True, exist_ok=True)
cache_source = os.environ.get('OBSERVERBENCH_APPS_CACHE')

if not CACHE_TARGET.exists():
    if cache_source:
        source = Path(cache_source).expanduser()
        if not source.is_file():
            raise FileNotFoundError(source)
    else:
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError(
                'Set OBSERVERBENCH_APPS_CACHE to the frozen followup_cache.npz'
            ) from error
        print('Upload followup_cache.npz (expected size: about 56 MB).')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Upload exactly one frozen cache file.')
        source = Path(next(iter(uploaded)))
    if source.resolve() != CACHE_TARGET.resolve():
        shutil.copy2(source, CACHE_TARGET)

print('Frozen activation cache:', CACHE_TARGET, CACHE_TARGET.stat().st_size, 'bytes')


## 3. Run once

The configuration pins the model/cache correspondence, SAE revision and file hash, feature-count and regularization grids, split, selection metric, audit budgets, and bootstrap seeds. Results can persist to Drive; rerunning with `--resume` reuses a hash-matched encoded SAE cache.


In [ ]:
PERSIST_TO_DRIVE = True
if PERSIST_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTDIR = Path('/content/drive/MyDrive/observerbench/qwen_apps_sae_observer_v1')
    except ImportError:
        OUTDIR = REPO_ROOT / 'results/revision/ai_control/controlarena_apps_qwen_sae_probe_v1'
else:
    OUTDIR = REPO_ROOT / 'results/revision/ai_control/controlarena_apps_qwen_sae_probe_v1'
OUTDIR.mkdir(parents=True, exist_ok=True)

CONFIG = REPO_ROOT / 'configs/revision/ai_control/controlarena_apps_qwen_sae_probe_v1.json'
command = [
    sys.executable,
    str(REPO_ROOT / 'scripts/run_controlarena_apps_sae_observer.py'),
    '--config', str(CONFIG),
    '--outdir', str(OUTDIR),
    '--download-checkpoint',
    '--device', 'cuda',
    '--resume',
]
print('Output directory:', OUTDIR)
subprocess.run(command, cwd=REPO_ROOT, check=True)


## 4. Read the locked-test result

The table below is the existing ObserverBench decision contract at the primary 2% audit budget. It compares the sparse SAE observer with a raw-residual logistic control and a logistic probe over the SAE reconstruction. Statistical AUROC and realized violations remain separate columns.


In [ ]:
import pandas as pd

result = json.loads((OUTDIR / 'results.json').read_text())
primary = result['budget_results']['0.02']
rows = []
for monitor, record in primary.items():
    metrics = record['metrics']
    rows.append({
        'observer': monitor,
        'risk_auroc': metrics['risk_auroc'],
        'realized_violations': metrics['realized_violations'],
        'audit_precision': metrics['audit_precision'],
        'n_audited': record['n_audited'],
    })
pd.DataFrame(rows).sort_values(['realized_violations', 'risk_auroc'], ascending=[True, False])


In [ ]:
diagnostics = json.loads((OUTDIR / 'context_length_diagnostics.json').read_text())
print('Rows above the SAE training context:', diagnostics['locked_rows']['n_above_training_context'])
print('SAE reconstruction diagnostics:')
display(pd.DataFrame(result['reconstruction_diagnostics']).T)
print('Selected sparse-probe hyperparameters:')
selection = result['selections']['qwen-neutral-sae-sparse-probe']
display({key: selection[key] for key in ('feature_count', 'c', 'calibration_metrics')})


## 5. Export the submission bundle

Each observer directory contains the target-free `predictions.csv` contract and its `observer_card.json`. The archive also carries the frozen configuration, locked results, length check, prevalence sweep, and paired bootstrap.


In [ ]:
archive = shutil.make_archive(str(OUTDIR), 'gztar', root_dir=OUTDIR)
print('Reproduction bundle:', archive)
